In [8]:
import numpy as np
import matplotlib.pyplot as plt

from external.pykernels.pykernels.basic import Linear, RBF, Polynomial

In [ ]:
# documentary comment: function to approximate is a  vectorised function
# note that this can be done way more easily with scikit learn as a black box funciton
# TODO das ganze für 2D workking machen

def sin_function(x):
    return np.sin(2*x+3)-1

# if noise_index was 1 then same magnitude of noise as signal - regression difficult
def gaussian_noise(y_values, noise_index, rng=None):
    std_noise = noise_index * np.std(y_values)
    gaussian_noise = rng.normal(0, std_noise, len(y_values))
    return gaussian_noise, std_noise

# def kernel_function(x1, x2, kernel='RBF'):
    if kernel == 'Polynomial':
        return 


In [ ]:
class GPR_regression:
    # TODO change kernel functiomn default
    def __init__(self, kernel=RBF()):
        self.c = None
        self.kernel_function = kernel

    def fit(self, x_train, y_train):

    def predict(self, X_test):
        # dummy labels for dataset, this is not considered during testing
        y_dummy = np.zeros(len(X_test))
        testset = self.dataset_class(X_test, y_dummy)
        test_loader = DataLoader(testset, batch_size = self.batch_size, shuffle = False, num_workers = self.num_workers)

        if self.trained_model is not None:
            y_pred = test_without_labels(self.trained_model, test_loader)
            return y_pred
        else:
            print("The model has not been trained before trying to predict!")
            return None
        

    def evaluate(self, X_test, y_true, metric = METRIC, verbose=False):
        # estimate a metric on the given set
        # this method does not use the same metric as test() but rather the classic metric that we know to make it comparable with the other
        y_pred = self.predict(X_test)
        if y_pred is None:
            raise ValueError("Prediction cannot be None. Is the model trained?")
        met = metric(y_true, y_pred)

        if verbose:
            print(f"Achieved {met}")
        return met

In [ ]:
def create_noised_train_set(function_to_approximate, noise_index=0.1, lower_boundary=-5, upper_boundary=5, number_of_training_samples=100, rng=None):
    x_all = rng.uniform(lower_boundary, upper_boundary, size=1000)
    y_without_noise = function_to_approximate(x_all)
    y_all = y_without_noise + gaussian_noise(y_without_noise, noise_index)


    training_indices = np.random.choice(len(x_all), size=number_of_training_samples, random_state=rng_seed, replace=False) # TODO das ist doppelt gemoppelt, 
    x_train = x_all[training_indices]
    y_train = y_all[training_indices]

    return x_train, y_train

In [ ]:
# class used to train a neural network with training data and predict class labels of new samples
class NeuralNetworkClassifier:
    def __init__(self, dataset_class, num_in, num_out, hidden_size=50, num_hlayers=3, add_norms=False, activation_function = nn.GELU, dropout=None):
        # initialises a new classifier already specifying the used model
        self.dataset_class = dataset_class
        self.model = Simple_MLP(num_in = num_in, 
               num_out = num_out,
               hidden_size = hidden_size,
               num_hlayers=num_hlayers, 
               add_norms=add_norms, activation_function=activation_function, dropout=dropout)
        self.num_workers = 0
        self.trained_model = None
        self.num_in = num_in
        self.num_out = num_out  
        self.hidden_size = hidden_size
        self.num_hlayers = num_hlayers
        self.activation_function = activation_function
        self.dropout = dropout
        self.add_norms = add_norms
    
    # one could make this tidier by a method "set training params"
    def fit(self, X_train, y_train, metric = None, batch_size=32, early_stopping = False, X_val = None, y_val = None, loss_fn = nn.CrossEntropyLoss(),  optimizer = None, num_epochs = 100):
        # this method trains the neural network specified in __init__ with the parameters given to fit
        assert self.num_in == X_train.shape[1], "The test samples have a different number of features as the model specified by __init__ has input neurons!"
        assert self.num_out == len(np.unique(y_train))

        if self.trained_model is not None: # reinitialize the model so the previous training does not affect this training loop
            self.model = Simple_MLP(num_in = self.num_in, 
               num_out = self.num_out,
               hidden_size = self.hidden_size,
               num_hlayers=self.num_hlayers, 
               add_norms=self.add_norms, activation_function=self.activation_function, dropout=self.dropout)

        if optimizer is None:
            optimizer = torch.optim.AdamW(self.model.parameters(),lr=0.01)

        if metric is None:
            metric = Accuracy(task="multiclass", num_classes=self.num_out, average="macro") # the default here is to used balanced accuracy

        self.batch_size = batch_size
        trainset = self.dataset_class(X_test, y_train) # create a new dataset
        test_loader = DataLoader(trainset, batch_size = self.batch_size, shuffle = True, num_workers = self.num_workers) # create a new dataloader containing the dataset

        if (X_val is not None and y_val is not None and early_stopping):
            # then we train the neural network using early stopping with the validation dataset
            valset = self.dataset_class(X_val, y_val)
            val_loader = DataLoader(valset, batch_size = self.batch_size, shuffle = True, num_workers = self.num_workers)
            self.trained_model, _, _ = train_neural_network(self.model, train_loader, optimizer, metric, loss_fn = loss_fn, val_loader = val_loader, num_epochs = num_epochs, early_stopping = True)
        else:
            self.trained_model, _ = train_neural_network(self.model, train_loader, optimizer, metric, num_epochs = num_epochs, loss_fn = loss_fn, early_stopping = False)

    def predict(self, X_test):
        # predicts class labels for (unseen), unlabeled samples
        assert self.num_in == X_test.shape[1], "The test samples have a different number of features as the trained model!"

        # dummy labels for dataset, this is not considered during testing
        y_dummy = np.zeros(len(X_test))
        testset = self.dataset_class(X_test, y_dummy)
        test_loader = DataLoader(testset, batch_size = self.batch_size, shuffle = False, num_workers = self.num_workers)

        if self.trained_model is not None:
            y_pred = test_without_labels(self.trained_model, test_loader)
            return y_pred
        else:
            print("The model has not been trained before trying to predict!")
            return None

    def evaluate(self, X_test, y_true, metric = METRIC, verbose=False):
        # estimate a metric on the given set
        # this method does not use the same metric as test() but rather the classic metric that we know to make it comparable with the other
        # classifiers in the ensemble classifier
        y_pred = self.predict(X_test)
        if y_pred is None:
            raise ValueError("Classifier prediction cannot be None. Is the model trained?")
        met = metric(y_true, y_pred)

        if verbose:
            print(f"Achieved {met}")
        return met